# 07 — Evaluation der drei LLM-Pipelines

Vergleicht die Erklärungsqualität von:
- **Pipeline 04** (JSON → Text): strukturierte Daten als Input
- **Pipeline 05** (Vision → Text): Waterfall-Plot als Input
- **Pipeline 06** (Tool-Use): LLM fragt Daten aktiv ab

**Drei Bewertungsebenen:**
1. Quantitativ (kein API-Key nötig): Länge, Token-Verbrauch, Kosten, Laufzeit
2. Faithfulness-Check (kein API-Key): Erwähnt die Erklärung die tatsächlich wichtigsten Features?
3. LLM-as-Judge (API-Key nötig): Strukturierte Bewertung nach Faithfulness, Verständlichkeit, Vollständigkeit

In [1]:
from __future__ import annotations

import sys, json, re, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display

from utils import INSTANCE_IDS, RESULTS_DIR, EXPLANATIONS_DIR
from utils.llm import ask_text, DEFAULT_MODEL

LOSS_KEY = 'poisson_log'
MODEL    = DEFAULT_MODEL
PIPELINES = ['04', '05', '06']
PIPELINE_LABELS = {'04': 'JSON→Text', '05': 'Vision', '06': 'Tool-Use'}
XAI_MODELS = ['xgb', 'ebm']

# Kosten pro 1M Token (claude-sonnet-4-6, Stand Mai 2025)
COST_INPUT_PER_M       = 3.00   # USD – reguläre Input-Tokens
COST_CACHE_READ_PER_M  = 0.30   # USD – Cache-Read-Tokens (90 % Rabatt)
COST_OUTPUT_PER_M      = 15.00  # USD – Output-Tokens

print('Setup abgeschlossen.')

Setup abgeschlossen.


## 1 · Ergebnisse laden

In [2]:
records = []
for pipeline in PIPELINES:
    p = RESULTS_DIR / f'pipeline{pipeline}'
    for xai in XAI_MODELS:
        for iid in INSTANCE_IDS:
            f = p / f'{xai}_inst{iid}.json'
            if not f.exists():
                print(f'FEHLT: {f}')
                continue
            d = json.loads(f.read_text())
            usage = d.get('usage', {})
            in_tok   = usage.get('input_tokens', 0)
            out_tok  = usage.get('output_tokens', 0)
            cache_r  = usage.get('cache_read_input_tokens', 0)
            # Cache-Read-Tokens kosten nur $0.30/M statt $3.00/M
            regular_in = max(in_tok - cache_r, 0)
            cost = (
                regular_in * COST_INPUT_PER_M
                + cache_r  * COST_CACHE_READ_PER_M
                + out_tok  * COST_OUTPUT_PER_M
            ) / 1_000_000
            records.append({
                'pipeline':      pipeline,
                'pipeline_label': PIPELINE_LABELS[pipeline],
                'xai_model':     xai.upper(),
                'instance_id':   iid,
                'explanation':   d.get('explanation', ''),
                'word_count':    len(d.get('explanation', '').split()),
                'tok_input':     in_tok,
                'tok_output':    out_tok,
                'tok_cache':     cache_r,
                'tok_total':     in_tok + out_tok,
                'cost_usd':      round(cost, 5),
                'elapsed_s':     d.get('elapsed_s', 0),
                'n_tool_calls':  d.get('n_tool_calls', 0),
                'y_true':        d.get('y_true', None),
                'prediction':    d.get('prediction', None),
            })

df = pd.DataFrame(records)
print(f'{len(df)} Ergebnisse geladen.')
display(df.groupby(['pipeline_label', 'xai_model']).size().unstack())

60 Ergebnisse geladen.


xai_model,EBM,XGB
pipeline_label,,
JSON→Text,10,10
Tool-Use,10,10
Vision,10,10


In [3]:
# Instanzen-Stichprobe dokumentieren
# Stratifizierte Auswahl über cnt-Quintile (keine Extremfälle: cnt>=31, kein Gewitter, kein Feiertagscluster)
instance_doc = [
    {"instance_id": 224,  "cnt_true":  51, "note": "Do 02M 13h, klar, ~8°C, 2011"},
    {"instance_id": 580,  "cnt_true": 159, "note": "So 03M 17h, klar, ~13°C, 2011"},
    {"instance_id": 1041, "cnt_true": 251, "note": "So 05M 10h, bewölkt, ~27°C, 2011"},
    {"instance_id": 1481, "cnt_true": 114, "note": "Sa 07M 08h, klar, ~32°C, 2011"},
    {"instance_id": 1677, "cnt_true": 410, "note": "Fr 08M 18h, bewölkt, ~30°C, 2011"},
    {"instance_id": 2058, "cnt_true":  31, "note": "Fr 10M 05h, klar, ~14°C, 2011"},
    {"instance_id": 2510, "cnt_true":  89, "note": "Mi 12M 10h, bewölkt, ~20°C, 2011"},
    {"instance_id": 3543, "cnt_true": 191, "note": "So 05M 09h, bewölkt, ~21°C, 2012"},
    {"instance_id": 3847, "cnt_true": 302, "note": "So 06M 20h, klar, ~25°C, 2012"},
    {"instance_id": 4454, "cnt_true": 557, "note": "Mi 09M 07h, klar, ~21°C, 2012"},
]
inst_df = pd.DataFrame(instance_doc)
print("Ausgewählte Test-Instanzen (stratifiziertes Quantil-Sampling):")
print("  cnt-Bereich: 31–557, verteilt über 5 cnt-Quintile")
print("  Kein cnt<=20, kein Gewitter (weathersit=4), beide Jahre (2011/2012)")
print()
display(inst_df)


Ausgewählte Test-Instanzen (stratifiziertes Quantil-Sampling):
  cnt-Bereich: 31–557, verteilt über 5 cnt-Quintile
  Kein cnt<=20, kein Gewitter (weathersit=4), beide Jahre (2011/2012)



,instance_id,cnt_true,note
0,224,51,"Do 02M 13h, klar, ~8°C, 2011"
1,580,159,"So 03M 17h, klar, ~13°C, 2011"
2,1041,251,"So 05M 10h, bewölkt, ~27°C, 2011"
3,1481,114,"Sa 07M 08h, klar, ~32°C, 2011"
4,1677,410,"Fr 08M 18h, bewölkt, ~30°C, 2011"
5,2058,31,"Fr 10M 05h, klar, ~14°C, 2011"
6,2510,89,"Mi 12M 10h, bewölkt, ~20°C, 2011"
7,3543,191,"So 05M 09h, bewölkt, ~21°C, 2012"
8,3847,302,"So 06M 20h, klar, ~25°C, 2012"
9,4454,557,"Mi 09M 07h, klar, ~21°C, 2012"


## 2 · Quantitative Analyse

In [4]:
quant = df.groupby('pipeline_label').agg(
    Wörter_mean  =('word_count',  'mean'),
    Wörter_std   =('word_count',  'std'),
    tok_input    =('tok_input',   'mean'),
    tok_output   =('tok_output',  'mean'),
    tok_total    =('tok_total',   'mean'),
    Kosten_USD   =('cost_usd',    'sum'),
    Zeit_s       =('elapsed_s',   'mean'),
    Tool_Calls   =('n_tool_calls','mean'),
).round(2)
print('Quantitativer Vergleich (Mittelwert pro Pipeline):')
display(quant)

Quantitativer Vergleich (Mittelwert pro Pipeline):


,Wörter_mean,Wörter_std,tok_input,tok_output,tok_total,Kosten_USD,Zeit_s,Tool_Calls
pipeline_label,,,,,,,,
JSON→Text,210.75,9.90,1837.15,510.75,2347.90,0.26,12.17,0.00
Tool-Use,305.85,22.36,3427.00,1262.90,4689.90,0.58,29.78,5.85
Vision,207.05,8.46,2186.60,512.45,2699.05,0.28,12.33,0.00


In [5]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
labels = [PIPELINE_LABELS[p] for p in PIPELINES]
colors = ['#4c72b0', '#dd8452', '#55a868']

# Wortanzahl
vals = [df[df.pipeline == p]['word_count'].mean() for p in PIPELINES]
axes[0].bar(labels, vals, color=colors)
axes[0].set_title('Ø Wortanzahl')
axes[0].set_ylabel('Wörter')

# Token-Verbrauch
in_vals  = [df[df.pipeline == p]['tok_input'].mean()  for p in PIPELINES]
out_vals = [df[df.pipeline == p]['tok_output'].mean() for p in PIPELINES]
x = range(len(labels))
axes[1].bar(x, in_vals,  label='Input',  color=colors, alpha=0.5)
axes[1].bar(x, out_vals, label='Output', color=colors, bottom=in_vals)
axes[1].set_xticks(list(x)); axes[1].set_xticklabels(labels)
axes[1].set_title('Ø Token-Verbrauch')
axes[1].set_ylabel('Tokens')
axes[1].legend()

# Kosten gesamt
vals = [df[df.pipeline == p]['cost_usd'].sum() for p in PIPELINES]
axes[2].bar(labels, vals, color=colors)
axes[2].set_title('Gesamtkosten (10 Calls)')
axes[2].set_ylabel('USD')

plt.suptitle('Quantitativer Vergleich der drei LLM-Pipelines', y=1.02)
plt.tight_layout()
out_path = RESULTS_DIR / 'eval_quantitative.png'
plt.savefig(out_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Gespeichert: {out_path}')

Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_1205/results/eval_quantitative.png


/var/folders/z8/rlz8vq9j2cs6wj25qj_g6y700000gn/T/ipykernel_93790/4182671026.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3 · Faithfulness-Check (keyword-basiert)

In [6]:
# Mapping: Feature-Name → deutsche Synonyme, die in der Erklärung auftauchen können
FEATURE_KEYWORDS: dict[str, list[str]] = {
    "hr":         ["uhrzeit", "stunde", "tageszeit", "uhr", "morgen", "abend", "nacht", "mittag"],
    "temp":       ["temperatur", "wärme", "warm", "kalt", "grad", "°c"],
    "yr":         ["jahr", "2011", "2012", "wachstum", "trend"],
    "mnth":       ["monat", "januar", "februar", "märz", "april", "mai", "juni",
                   "juli", "august", "september", "oktober", "november", "dezember",
                   "sommer", "winter", "frühling", "herbst"],
    "weekday":    ["wochentag", "montag", "dienstag", "mittwoch", "donnerstag",
                   "freitag", "samstag", "sonntag", "wochenende", "werktag"],
    "weathersit": ["wetter", "regen", "bewölkt", "sonnig", "klar", "nebel", "schnee"],
    "hum":        ["luftfeuchtigkeit", "feuchtigkeit", "humid"],
    "windspeed":  ["wind", "windgeschwindigkeit", "sturm", "brise"],
    "holiday":    ["feiertag", "urlaub", "frei"],
}


def top_features_from_json(xai_model: str, instance_id: int, top_k: int = 3) -> list[str]:
    path = EXPLANATIONS_DIR / f"local_{xai_model.lower()}_{LOSS_KEY}_inst{instance_id}.json"
    d = json.loads(path.read_text())
    return [c["feature"] for c in d["contributions"][:top_k]]


def mentions_feature(text: str, feature: str) -> bool:
    text_lower = text.lower()
    keywords = FEATURE_KEYWORDS.get(feature, [feature])
    return any(kw in text_lower for kw in keywords)


faith_rows = []
for _, row in df.iterrows():
    top3 = top_features_from_json(row["xai_model"], row["instance_id"])
    hits = sum(mentions_feature(row["explanation"], f) for f in top3)
    faith_rows.append({
        "pipeline_label": row["pipeline_label"],
        "xai_model":      row["xai_model"],
        "instance_id":    row["instance_id"],
        "top3_features":  top3,
        "mentioned":      hits,
        "faithfulness":   round(hits / 3, 2),
    })

faith_df = pd.DataFrame(faith_rows)

faith_summary = faith_df.groupby("pipeline_label")["faithfulness"].agg(["mean", "std"]).round(3)
faith_summary.columns = ["Ø Faithfulness", "Std"]
print("Faithfulness (0–1): Anteil der Top-3-Features erwähnt:")
display(faith_summary)

print()
print("Details:")
display(faith_df[["pipeline_label","xai_model","instance_id","top3_features","mentioned","faithfulness"]])


Faithfulness (0–1): Anteil der Top-3-Features erwähnt:


,Ø Faithfulness,Std
pipeline_label,,
JSON→Text,1.000,0.000
Tool-Use,0.984,0.074
Vision,1.000,0.000



Details:


,pipeline_label,xai_model,instance_id,top3_features,mentioned,faithfulness
0,JSON→Text,XGB,224,"[hr, yr, temp]",3,1.00
1,JSON→Text,XGB,580,"[hr, mnth, temp]",3,1.00
2,JSON→Text,XGB,1041,"[weekday, temp, yr]",3,1.00
3,JSON→Text,XGB,1481,"[hr, yr, weekday]",3,1.00
4,JSON→Text,XGB,1677,"[yr, temp, weekday]",3,1.00
5,JSON→Text,XGB,2058,"[yr, hr, hum]",3,1.00
6,JSON→Text,XGB,2510,"[hr, yr, mnth]",3,1.00
7,JSON→Text,XGB,3543,"[hr, temp, yr]",3,1.00
8,JSON→Text,XGB,3847,"[hr, yr, weekday]",3,1.00
9,JSON→Text,XGB,4454,"[hr, yr, temp]",3,1.00


## 4 · LLM-as-Judge Evaluation

> **Voraussetzung:** `ANTHROPIC_API_KEY` in `.env`.

Claude bewertet jede Erklärung auf drei Dimensionen (1–5):
- **Faithfulness**: Werden die wichtigsten Treiber korrekt beschrieben?
- **Clarity**: Ist die Erklärung für Nicht-Experten verständlich?
- **Completeness**: Werden Vorhersage, Treiber und praktische Implikation abgedeckt?

In [7]:
JUDGE_SYSTEM = """Du bewertest Erklärungen von Machine-Learning-Modellen für einen Fahrradverleih.
Bewerte jede Erklärung auf drei Kriterien (Skala 1–5) anhand der unten definierten
Rubrik. Antworte ausschließlich mit einem validen JSON-Objekt.

=== SCORING-RUBRIK ===

FAITHFULNESS (Treue zur Modellvorhersage):
  5 – Alle Top-3-Treiber korrekt genannt, Wirkungsrichtung stimmt,
      Vorhersage-Zahlenwert korrekt.
  4 – Mindestens 2 Top-3-Treiber korrekt; kleine Ungenauigkeiten erlaubt.
  3 – Mindestens 1 Top-3-Treiber korrekt; ein Treiber fehlt oder Richtung falsch.
  2 – Treiber nur vage beschrieben oder Wirkungsrichtung mehrfach falsch.
  1 – Kein Top-3-Treiber erkennbar oder massive Fehlinformationen.

  Abzüge:
    -1: Genannter Treiber nicht unter Top-3 (Halluzination)
    -1: Wirkungsrichtung eines Top-3-Treibers falsch
    -1: Vorhergesagter Zahlenwert fehlt völlig

CLARITY (Verständlichkeit für Nicht-Experten):
  5 – Kein Fachjargon, klare Alltagssprache, logischer Aufbau.
  4 – Weitgehend verständlich; ein Fachbegriff oder leicht unklar.
  3 – Mehrere Fachbegriffe oder unklare Passagen; Laie muss raten.
  2 – Überwiegend technische Sprache; schwer zugänglich.
  1 – Unverständlich oder stark fehlerhaft.

  Abzüge:
    -1: Verwendung von "SHAP", "Log-Raum", "exp()" oder ähnlichem Fachjargon
    -1: Fehlende Alltagsübersetzung von normalisierten Werten (z.B. "temp=0.68" statt "~28°C")

COMPLETENESS (Vollständigkeit der drei Pflichtabschnitte):
  5 – Alle drei Abschnitte vorhanden und substanziell: Vorhersage, Treiber,
      praktische Betriebsempfehlung.
  4 – Alle drei vorhanden; ein Abschnitt nur kurz/oberflächlich.
  3 – Nur zwei Abschnitte erkennbar oder einer sehr schwach.
  2 – Vorhersage fehlt oder Empfehlung fehlt; nur Treiber beschrieben.
  1 – Strukturlos; keiner der Pflichtabschnitte erkennbar.

  Abzüge:
    -1: Kein Vergleich Vorhersage vs. tatsächlicher Wert
    -1: Keine praktische Implikation / Betriebsempfehlung"""


WEEKDAYS_JUDGE = {0:"Sonntag", 1:"Montag", 2:"Dienstag", 3:"Mittwoch",
                  4:"Donnerstag", 5:"Freitag", 6:"Samstag"}
MONTHS_JUDGE   = {1:"Januar", 2:"Februar", 3:"März", 4:"April", 5:"Mai",
                  6:"Juni", 7:"Juli", 8:"August", 9:"September",
                  10:"Oktober", 11:"November", 12:"Dezember"}
WEATHER_JUDGE  = {1:"klar/wenige Wolken", 2:"Nebel/bewölkt",
                  3:"leichter Regen/Schnee", 4:"Starkregen/Gewitter"}


def build_judge_prompt(row: dict, xai_model: str, instance_id: int) -> str:
    local_path = EXPLANATIONS_DIR / f"local_{xai_model.lower()}_{LOSS_KEY}_inst{instance_id}.json"
    l = json.loads(local_path.read_text())
    fv = l["feature_values"]

    top3 = [{"feature": c["feature"], "contribution": c["contribution"],
              "value": c["value"]}
             for c in l["contributions"][:3]]

    # Menschenlesbare Feature-Werte für den Judge
    fv_readable = {
        "uhrzeit":           f"{int(fv['hr']):02d}:00 Uhr",
        "wochentag":         WEEKDAYS_JUDGE.get(int(fv["weekday"]), str(fv["weekday"])),
        "monat":             MONTHS_JUDGE.get(int(fv["mnth"]), str(fv["mnth"])),
        "jahr":              "2011" if int(fv["yr"]) == 0 else "2012",
        "wetter":            WEATHER_JUDGE.get(int(fv["weathersit"]), str(fv["weathersit"])),
        "temperatur_celsius": f"~{float(fv['temp']) * 41:.1f} °C",
        "luftfeuchtigkeit":  f"{float(fv['hum']) * 100:.0f} %",
        "windgeschwindigkeit": f"{float(fv['windspeed']) * 67:.1f} km/h",
        "feiertag":          "ja" if int(fv["holiday"]) == 1 else "nein",
    }

    ground_truth = {
        "model":                  xai_model,
        "prediction":             l["prediction"],
        "y_true":                 l["y_true"],
        "top3_drivers":           top3,
        "feature_values_readable": fv_readable,
    }

    # Für Tool-Use: Hinweis dass Tool-Outputs korrekt sind
    pipeline = row.get("pipeline", "")
    if "06" in pipeline or pipeline == "06_tooluse":
        ground_truth["tool_context"] = (
            "Hinweis: Diese Erklärung wurde von einem Tool-Use-Agenten generiert. "
            "Zahlenwerte aus Tool-Calls (Partial-Dependence-Kurven, Percentile, "
            "kontrafaktische Vorhersagen) sind korrekt und dürfen als Belege "
            "für Faithfulness gewertet werden — auch wenn sie nicht explizit in "
            "top3_drivers stehen."
        )

    return json.dumps({
        "task": (
            "Bewerte die folgende Erklärung nach der definierten Rubrik. "
            "Vergib für jedes Kriterium einen Score (1–5) und begründe kurz."
        ),
        "ground_truth": ground_truth,
        "explanation": row["explanation"],
        "output_format": {
            "faithfulness": "<int 1–5>",
            "clarity": "<int 1–5>",
            "completeness": "<int 1–5>",
            "reasoning": {
                "faithfulness": "<1 Satz>",
                "clarity": "<1 Satz>",
                "completeness": "<1 Satz>",
            },
        },
    }, ensure_ascii=False, indent=2)


In [8]:
# Ergebnisse aus Cache laden falls vorhanden (v1-Judge-Ergebnisse)
_v1_cache = RESULTS_DIR / 'eval_llm_judge.json'
if _v1_cache.exists():
    judge_df = pd.DataFrame(json.loads(_v1_cache.read_text()))
    for col in ['faithfulness', 'clarity', 'completeness']:
        judge_df[col] = pd.to_numeric(judge_df[col], errors='coerce')
    print(f'v1-Ergebnisse aus Cache geladen: {len(judge_df)} Einträge')
    print('(Zelle überspringt API-Calls – entferne die Cache-Datei zum Neuberechnen)')

# Alle 30 Judge-Calls ausführen:
judge_rows = []
total_in, total_out = 0, 0

for _, row in df.iterrows():
    prompt = build_judge_prompt(
        row.to_dict(), row['xai_model'], row['instance_id']
    )
    response = ask_text(
        prompt,
        system=JUDGE_SYSTEM,
        model=MODEL,
        max_tokens=400,
        cache_system=True,
    )
    usage   = response.get('usage', {})
    in_tok  = usage.get('input_tokens', 0)
    out_tok = usage.get('output_tokens', 0)
    total_in += in_tok; total_out += out_tok

    raw = response['content'][0]['text'].strip()
    # JSON aus Antwort extrahieren (robust gegen Markdown-Codeblöcke)
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    scores = json.loads(m.group()) if m else {}

    judge_rows.append({
        'pipeline_label': row['pipeline_label'],
        'xai_model':      row['xai_model'],
        'instance_id':    row['instance_id'],
        'faithfulness':   scores.get('faithfulness', None),
        'clarity':        scores.get('clarity', None),
        'completeness':   scores.get('completeness', None),
        'reasoning':      scores.get('reasoning', ''),
        'raw_response':   raw,
    })
    print(f"  {row['pipeline_label']:12s} {row['xai_model']} inst={row['instance_id']:4d}  "
          f"F={scores.get('faithfulness','?')} "
          f"C={scores.get('clarity','?')} "
          f"Co={scores.get('completeness','?')}  in={in_tok}")

judge_df = pd.DataFrame(judge_rows)
# Ergebnisse speichern
(RESULTS_DIR / 'eval_llm_judge.json').write_text(
    json.dumps(judge_rows, indent=2, ensure_ascii=False)
)
print(f'\nGesamt: input={total_in}  output={total_out}')

v1-Ergebnisse aus Cache geladen: 60 Einträge
(Zelle überspringt API-Calls – entferne die Cache-Datei zum Neuberechnen)
  JSON→Text    XGB inst= 224  F=3 C=5 Co=4  in=1783
  JSON→Text    XGB inst= 580  F=5 C=5 Co=5  in=1769
  JSON→Text    XGB inst=1041  F=5 C=5 Co=5  in=1805
  JSON→Text    XGB inst=1481  F=5 C=5 Co=5  in=1812
  JSON→Text    XGB inst=1677  F=4 C=4 Co=5  in=1807
  JSON→Text    XGB inst=2058  F=4 C=5 Co=5  in=1786
  JSON→Text    XGB inst=2510  F=4 C=4 Co=5  in=1790
  JSON→Text    XGB inst=3543  F=4 C=4 Co=5  in=1818
  JSON→Text    XGB inst=3847  F=4 C=4 Co=5  in=1811
  JSON→Text    XGB inst=4454  F=5 C=5 Co=5  in=1843
  JSON→Text    EBM inst= 224  F=5 C=5 Co=5  in=1728
  JSON→Text    EBM inst= 580  F=4 C=5 Co=5  in=1763
  JSON→Text    EBM inst=1041  F=5 C=5 Co=5  in=1798
  JSON→Text    EBM inst=1481  F=5 C=5 Co=5  in=1787
  JSON→Text    EBM inst=1677  F=5 C=5 Co=5  in=1790
  JSON→Text    EBM inst=2058  F=5 C=5 Co=5  in=1801
  JSON→Text    EBM inst=2510  F=5 C=5 Co=5  in=18

## 5 · Ergebnisse des LLM-Judges

In [9]:
# Fehlende Scores dokumentieren (JSON-Parse-Fehler beim Judge-Output)
null_mask = judge_df[['faithfulness', 'clarity', 'completeness']].isnull().any(axis=1)
if null_mask.any():
    print(f"⚠️  {null_mask.sum()} Einträge mit None-Scores (JSON-Parsing fehlgeschlagen):")
    display(judge_df[null_mask][['pipeline_label', 'xai_model', 'instance_id']])
    print("   → Diese Einträge werden von .mean() automatisch ausgeschlossen (pandas NaN-Handling).")
    print()

judge_summary = judge_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']].mean().round(2)
judge_counts  = judge_df.groupby('pipeline_label')[['faithfulness']].count().rename(
    columns={'faithfulness': 'n (gültig)'}
)
judge_summary = judge_summary.join(judge_counts)
print('Ø Judge-Scores (1–5) pro Pipeline  [None-Einträge aus Mittelwert ausgeschlossen]:')
display(judge_summary)

print()
judge_xai = judge_df.groupby(['pipeline_label', 'xai_model'])[['faithfulness', 'clarity', 'completeness']].mean().round(2)
print('Aufgeschlüsselt nach Modell:')
display(judge_xai)


⚠️  7 Einträge mit None-Scores (JSON-Parsing fehlgeschlagen):


,pipeline_label,xai_model,instance_id
31,Vision,EBM,580
33,Vision,EBM,1481
35,Vision,EBM,2058
41,Tool-Use,XGB,580
48,Tool-Use,XGB,3847
55,Tool-Use,EBM,2058
59,Tool-Use,EBM,4454


   → Diese Einträge werden von .mean() automatisch ausgeschlossen (pandas NaN-Handling).

Ø Judge-Scores (1–5) pro Pipeline  [None-Einträge aus Mittelwert ausgeschlossen]:


,faithfulness,clarity,completeness,n (gültig)
pipeline_label,,,,
JSON→Text,4.55,4.80,4.95,20
Tool-Use,4.62,4.44,5.00,16
Vision,4.24,4.76,4.88,17



Aufgeschlüsselt nach Modell:


faithfulness  clarity  completeness
pipeline_label xai_model                                     
JSON→Text      EBM                4.80     5.00          5.00
               XGB                4.30     4.60          4.90
Tool-Use       EBM                4.62     4.25          5.00
               XGB                4.62     4.62          5.00
Vision         EBM                3.86     4.71          4.86
               XGB                4.50     4.80          4.90

In [10]:
# Radar-Chart
from matplotlib.patches import FancyArrowPatch
import numpy as np

criteria   = ['Faithfulness', 'Clarity', 'Completeness']
n          = len(criteria)
angles     = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
angles    += angles[:1]
pipeline_colors = {'JSON→Text': '#4c72b0', 'Vision': '#dd8452', 'Tool-Use': '#55a868'}

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

for label, color in pipeline_colors.items():
    row = judge_summary.loc[label] if label in judge_summary.index else None
    if row is None: continue
    values = [row['faithfulness'], row['clarity'], row['completeness']]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=label, color=color)
    ax.fill(angles, values, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(criteria, fontsize=12)
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(['1','2','3','4','5'], fontsize=8)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.set_title('LLM-Judge Scores (1–5) pro Pipeline', pad=20)

plt.tight_layout()
out_path = RESULTS_DIR / 'eval_radar.png'
plt.savefig(out_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Gespeichert: {out_path}')

Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_1205/results/eval_radar.png


/var/folders/z8/rlz8vq9j2cs6wj25qj_g6y700000gn/T/ipykernel_93790/843577156.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6 · Gesamtübersicht und Empfehlung

In [11]:
# Kombinierte Tabelle: quantitativ + LLM-Judge
quant_p = df.groupby('pipeline_label').agg(
    Wörter    =('word_count', 'mean'),
    Tokens_in =('tok_input',  'mean'),
    Tokens_out=('tok_output', 'mean'),
    Kosten_USD=('cost_usd',   'sum'),
    Zeit_s    =('elapsed_s',  'mean'),
    ToolCalls =('n_tool_calls','mean'),
).round(2)

faith_p = faith_df.groupby('pipeline_label')['faithfulness'].mean().round(3)
faith_p.name = 'Faith_KW'

combined = quant_p.join(faith_p)
if len(judge_df) > 0:
    judge_p = judge_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']].mean().round(2)
    judge_p.columns = ['Judge_Faith', 'Judge_Clarity', 'Judge_Complete']
    judge_std = judge_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']].std().round(3)
    judge_std.columns = ['Judge_Faith_std', 'Judge_Clarity_std', 'Judge_Complete_std']
    judge_n = judge_df.groupby('pipeline_label')[['faithfulness']].count().rename(
        columns={'faithfulness': 'Judge_n'}
    )
    combined = combined.join(judge_p).join(judge_std).join(judge_n)

print('Gesamtübersicht:')
display(combined)

combined.to_csv(RESULTS_DIR / 'eval_summary.csv')
print(f'Gespeichert: {RESULTS_DIR / "eval_summary.csv"}')


Gesamtübersicht:


,Wörter,Tokens_in,Tokens_out,Kosten_USD,Zeit_s,ToolCalls,Faith_KW,Judge_Faith,Judge_Clarity,Judge_Complete,Judge_Faith_std,Judge_Clarity_std,Judge_Complete_std,Judge_n
pipeline_label,,,,,,,,,,,,,,
JSON→Text,210.75,1837.15,510.75,0.26,12.17,0.00,1.000,4.55,4.80,4.95,0.605,0.410,0.224,20
Tool-Use,305.85,3427.00,1262.90,0.58,29.78,5.85,0.984,4.62,4.44,5.00,0.500,0.512,0.000,16
Vision,207.05,2186.60,512.45,0.28,12.33,0.00,1.000,4.24,4.76,4.88,0.664,0.437,0.332,17


Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_1205/results/eval_summary.csv


In [12]:
print('=== EMPFEHLUNG ===' )
print()

if len(judge_df) > 0:
    judge_total = judge_df.groupby('pipeline_label')[['faithfulness','clarity','completeness']].mean().sum(axis=1)
    best_pipeline = judge_total.idxmax()
    print(f'Beste Pipeline nach LLM-Judge-Gesamtscore: {best_pipeline}')
    print()
    print('Score-Rangfolge:')
    for label, score in judge_total.sort_values(ascending=False).items():
        print(f'  {label:12s}: {score:.2f} / 15')
    print()

print('Trade-off-Analyse:')
print('  JSON→Text : niedrigster Token-Verbrauch, gute Faithfulness (Zahlen direkt verfügbar)')
print('  Vision    : mittlerer Verbrauch, LLM liest Beiträge aus Plot (kann unscharf sein)')
print('  Tool-Use  : höchster Verbrauch, LLM steuert selbst die Abfrage → potenziell beste')
print('              Vollständigkeit, da es gezielt nachhaken kann')

=== EMPFEHLUNG ===

Beste Pipeline nach LLM-Judge-Gesamtscore: JSON→Text

Score-Rangfolge:
  JSON→Text   : 14.30 / 15
  Tool-Use    : 14.06 / 15
  Vision      : 13.88 / 15

Trade-off-Analyse:
  JSON→Text : niedrigster Token-Verbrauch, gute Faithfulness (Zahlen direkt verfügbar)
  Vision    : mittlerer Verbrauch, LLM liest Beiträge aus Plot (kann unscharf sein)
  Tool-Use  : höchster Verbrauch, LLM steuert selbst die Abfrage → potenziell beste
              Vollständigkeit, da es gezielt nachhaken kann


## 8 · LLM-Judge v2 – Kalibrierter Prompt

**Verbesserungen gegenüber v1:**
- Explizite Scoring-Rubrik (1–5) mit Deduktionsregeln pro Kriterium
- `feature_values_readable` im Prompt: Judge kann genannte Zahlen gegen GT verifizieren
- Grundregel: Score 4 = Standard für korrekte/vollständige Erklärung; Score 5 nur ohne Abzüge

**Methodische Note:**
v2 ist für die Interpretation als primäre Quelle zu bevorzugen. v1 dient als Vergleich
und zeigt, dass unstrukturierte Prompts systematisch zu milden Urteilen führen.


In [13]:
import json
from pathlib import Path

# Immer initialisieren – verhindert NameError in Folge-Zellen
judge_v2_df = None

# v2-Ergebnisse laden
v2_path = RESULTS_DIR / 'eval_llm_judge_v2.json'
if not v2_path.exists():
    print('⚠️  eval_llm_judge_v2.json nicht gefunden – Zelle 11 (Judge-Calls) ausführen.')
else:
    judge_v2_df = pd.DataFrame(json.loads(v2_path.read_text()))
    for col in ['faithfulness', 'clarity', 'completeness']:
        judge_v2_df[col] = pd.to_numeric(judge_v2_df[col], errors='coerce')

    v2_summary = judge_v2_df.groupby('pipeline_label')[['faithfulness','clarity','completeness']].mean().round(2)
    v2_n = judge_v2_df.groupby('pipeline_label')[['faithfulness']].count().rename(columns={'faithfulness':'n'})
    v2_std = judge_v2_df.groupby('pipeline_label')[['faithfulness','clarity','completeness']].std().round(3)
    v2_std.columns = ['faith_std','clarity_std','complete_std']
    v2_full = v2_summary.join(v2_std).join(v2_n)
    print('v2 Judge-Scores (kalibrierter Prompt):')
    display(v2_full)

    # Ceiling-Effekt v2
    tv = judge_v2_df[['faithfulness','clarity','completeness']].count().sum()
    t5 = (judge_v2_df[['faithfulness','clarity','completeness']] == 5).sum().sum()
    t4 = (judge_v2_df[['faithfulness','clarity','completeness']] == 4).sum().sum()
    print(f'\nCeiling v2: {t5}/{tv} Scores=5 ({100*t5/tv:.0f}%),  '
          f'{t4}/{tv} Scores=4 ({100*t4/tv:.0f}%)')
    print(f'(v1 zum Vergleich: 79/87 Scores=5, 91 %)')

v2 Judge-Scores (kalibrierter Prompt):


,faithfulness,clarity,completeness,faith_std,clarity_std,complete_std,n
pipeline_label,,,,,,,
JSON→Text,3.8,4.7,5.0,0.919,0.483,0.000,10
Tool-Use,4.6,4.9,5.0,0.516,0.316,0.000,10
Vision,4.4,4.7,4.9,0.699,0.483,0.316,10



Ceiling v2: 66/90 Scores=5 (73%),  18/90 Scores=4 (20%)
(v1 zum Vergleich: 79/87 Scores=5, 91 %)


In [14]:
# Vergleich v1 vs v2: Faithfulness-Differenz (der interessanteste Befund)
if judge_v2_df is None:
    print('⚠️  judge_v2_df nicht verfügbar – zuerst Zelle 8 (v2-Laden) ausführen.')
else:
    v1_faith = judge_df.groupby('pipeline_label')['faithfulness'].mean().round(2)
    v2_faith = judge_v2_df.groupby('pipeline_label')['faithfulness'].mean().round(2)

    compare = pd.DataFrame({'v1_Faith': v1_faith, 'v2_Faith': v2_faith})
    compare['Δ (v2−v1)'] = (compare['v2_Faith'] - compare['v1_Faith']).round(2)
    compare['Interpretation'] = [
        'JSON→Text: v2 strenger (Ranking-Abweichungen erkannt)',
        'Tool-Use:  v2 fairer (hr-Werte nicht mehr als Halluzination gewertet)',
        'Vision:    v2 leicht strenger',
    ]

    # Tabelle
    print('Faithfulness v1 vs v2:')
    display(compare)

    # Balken-Diagramm: v1 vs v2 Faithfulness
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    labels = ['JSON→Text', 'Tool-Use', 'Vision']
    colors = ['#4c72b0', '#55a868', '#dd8452']
    x = range(len(labels))

    # Links: Faithfulness-Vergleich
    v1_vals = [judge_df[judge_df.pipeline_label==l]['faithfulness'].mean() for l in labels]
    v2_vals = [judge_v2_df[judge_v2_df.pipeline_label==l]['faithfulness'].mean() for l in labels]
    w = 0.35
    bars1 = axes[0].bar([i - w/2 for i in x], v1_vals, w, label='v1 (unkalibriert)', color=colors, alpha=0.5)
    bars2 = axes[0].bar([i + w/2 for i in x], v2_vals, w, label='v2 (kalibriert)', color=colors)
    axes[0].set_xticks(list(x)); axes[0].set_xticklabels(labels, rotation=10)
    axes[0].set_ylim(0, 5.5); axes[0].set_ylabel('Ø Faithfulness-Score (1–5)')
    axes[0].set_title('Faithfulness: v1 vs v2')
    axes[0].legend()
    for bars in [bars1, bars2]:
        for bar in bars:
            axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                         f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

    # Rechts: Score-Verteilung v2
    from collections import Counter
    all_scores_v2 = []
    for col in ['faithfulness','clarity','completeness']:
        all_scores_v2.extend(judge_v2_df[col].dropna().astype(int).tolist())
    cnt = Counter(all_scores_v2)
    score_vals = [cnt.get(s, 0) for s in range(1, 6)]
    axes[1].bar(range(1, 6), score_vals, color='#4c72b0')
    for i, v in enumerate(score_vals):
        axes[1].text(i+1, v+0.5, str(v), ha='center', fontsize=9)
    axes[1].set_xlabel('Score'); axes[1].set_ylabel('Anzahl')
    axes[1].set_title('Score-Verteilung v2 (alle 90 Einzelscores)')
    axes[1].set_xticks(range(1,6))

    plt.tight_layout()
    out = RESULTS_DIR / 'eval_judge_v1_v2_comparison.png'
    plt.savefig(out, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Gespeichert: {out}')

Faithfulness v1 vs v2:


,v1_Faith,v2_Faith,Δ (v2−v1),Interpretation
pipeline_label,,,,
JSON→Text,4.55,3.8,-0.75,JSON→Text: v2 strenger (Ranking-Abweichungen e...
Tool-Use,4.62,4.6,-0.02,Tool-Use: v2 fairer (hr-Werte nicht mehr als ...
Vision,4.24,4.4,0.16,Vision: v2 leicht strenger


Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_1205/results/eval_judge_v1_v2_comparison.png


/var/folders/z8/rlz8vq9j2cs6wj25qj_g6y700000gn/T/ipykernel_93790/3245954847.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9 · Judge v3 – Opus 4.7 (unabhängiges Modell)

**Motivation:** v1 und v2 verwendeten `claude-sonnet-4-6` sowohl zur Erklärungsgenerierung
als auch als Judge – bekannter Self-Preference-Bias (Panickssery et al., 2024).
v3 verwendet `claude-opus-4-7` als unabhängigen Judge mit identischer Kalibrierungsrubrik.

**Inhärente Einschränkung für Tool-Use:** Der Judge sieht nur `feature_values_readable`
und `top3_drivers` als Ground Truth – nicht die Tool-Call-Outputs (Partial-Dependence-Werte,
Feature-Statistiken). Zahlen, die die Tool-Use-Pipeline korrekt per Tool abgefragt hat
(z. B. „Baseline 190 Fahrräder", „−29 %"), gelten für den Judge als nicht-belegbar.
Dies führt zu strukturell niedrigeren Opus-Scores für Tool-Use, obwohl die Zahlen
methodisch korrekt sind. Der Effekt ist ein Messfehler des Judge-Setups, kein
Qualitätsmangel der Pipeline.


In [15]:
opus_path = RESULTS_DIR / 'eval_llm_judge_opus.json'
if not opus_path.exists():
    print('⚠️  eval_llm_judge_opus.json nicht gefunden.')
else:
    judge_opus_df = pd.DataFrame(json.loads(opus_path.read_text()))
    for col in ['faithfulness', 'clarity', 'completeness']:
        judge_opus_df[col] = pd.to_numeric(judge_opus_df[col], errors='coerce')

    opus_summary = judge_opus_df.groupby('pipeline_label')[['faithfulness','clarity','completeness']].mean().round(2)
    opus_std     = judge_opus_df.groupby('pipeline_label')[['faithfulness','clarity','completeness']].std().round(3)
    opus_std.columns = ['faith_std','clarity_std','complete_std']
    print('Opus 4.7 Judge-Scores (kalibrierter Prompt, unabhängiges Modell):')
    display(opus_summary.join(opus_std))

    # Ceiling Opus
    tv = judge_opus_df[['faithfulness','clarity','completeness']].count().sum()
    t5 = (judge_opus_df[['faithfulness','clarity','completeness']] == 5).sum().sum()
    t4 = (judge_opus_df[['faithfulness','clarity','completeness']] == 4).sum().sum()
    t3 = (judge_opus_df[['faithfulness','clarity','completeness']] <= 3).sum().sum()
    print(f'\nCeiling Opus: 5→{t5} ({100*t5/tv:.0f}%), 4→{t4} ({100*t4/tv:.0f}%), ≤3→{t3} ({100*t3/tv:.0f}%)')
    print('(v1 Sonnet: 91% / 7% / 2%  →  v2 Sonnet: 73% / 20% / 7%  →  Opus: 50% / 36% / 14%)')


Opus 4.7 Judge-Scores (kalibrierter Prompt, unabhängiges Modell):


,faithfulness,clarity,completeness,faith_std,clarity_std,complete_std
pipeline_label,,,,,,
JSON→Text,3.5,4.3,5.0,1.179,0.483,0.0
Tool-Use,3.7,3.8,5.0,1.059,0.422,0.0
Vision,4.4,4.1,5.0,0.699,0.568,0.0



Ceiling Opus: 5→45 (50%), 4→32 (36%), ≤3→13 (14%)
(v1 Sonnet: 91% / 7% / 2%  →  v2 Sonnet: 73% / 20% / 7%  →  Opus: 50% / 36% / 14%)


In [16]:
# Dreifach-Vergleich: v1 · v2 · Opus
if judge_v2_df is None or 'judge_opus_df' not in dir():
    print('⚠️  judge_v2_df oder judge_opus_df nicht verfügbar – zuerst Zellen 8 und 9 ausführen.')
else:
    labels_ord = ['JSON→Text', 'Tool-Use', 'Vision']
    judge_versions = [
        ('v1 Sonnet\n(unkalibriert)', judge_df,       'lightgrey', '#888'),
        ('v2 Sonnet\n(kalibriert)',   judge_v2_df,    '#7fbadf',   '#4c72b0'),
        ('v3 Opus\n(kalibriert)',     judge_opus_df,  '#4c72b0',   '#1a3a6e'),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax_i, (crit, title) in enumerate([('faithfulness', 'Faithfulness'),
                                           ('clarity',      'Clarity'),
                                           ('completeness', 'Completeness')]):
        x = np.arange(len(labels_ord)); w = 0.25
        for j, (vlabel, df_, face, edge) in enumerate(judge_versions):
            vals = [df_[df_.pipeline_label == l][crit].mean() for l in labels_ord]
            bars = axes[ax_i].bar(x + (j - 1) * w, vals, w, label=vlabel,
                                   color=face, edgecolor=edge, linewidth=0.8)
            for bar in bars:
                h = bar.get_height()
                axes[ax_i].text(bar.get_x() + bar.get_width() / 2, h + 0.06,
                                f'{h:.2f}', ha='center', fontsize=7)
        axes[ax_i].set_xticks(x); axes[ax_i].set_xticklabels(labels_ord, fontsize=9)
        axes[ax_i].set_ylim(0, 5.9); axes[ax_i].set_ylabel('Ø Score (1–5)', fontsize=9)
        axes[ax_i].set_title(title, fontsize=11)
        axes[ax_i].axhline(4, color='#aaa', linestyle='--', linewidth=0.7, alpha=0.6)
        if ax_i == 0:
            axes[ax_i].legend(fontsize=8, loc='lower right')

    plt.suptitle('LLM-as-Judge: v1 Sonnet (unkalibriert)  ·  v2 Sonnet (kalibriert)  ·  v3 Opus (kalibriert)',
                 y=1.02, fontsize=11)
    plt.tight_layout()
    out = RESULTS_DIR / 'eval_judge_all_versions.png'
    plt.savefig(out, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Gespeichert: {out}')

    print()
    print('Kernbefunde:')
    print('  Completeness: alle Judges einig → 5.0 (robuste Aussage)')
    print('  Faithfulness: Vision konsistent ~4.4; JSON→Text und Tool-Use divergieren je nach Judge')
    print('  Clarity Opus: Tool-Use 3.8 – Opus penalisiert Zahlen aus Tool-Calls als nicht-GT-belegbar')
    print('  → Opus-Niedrigscores für Tool-Use sind strukturelles Messartefakt, kein Qualitätsmangel')

Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_1205/results/eval_judge_all_versions.png

Kernbefunde:
  Completeness: alle Judges einig → 5.0 (robuste Aussage)
  Faithfulness: Vision konsistent ~4.4; JSON→Text und Tool-Use divergieren je nach Judge
  Clarity Opus: Tool-Use 3.8 – Opus penalisiert Zahlen aus Tool-Calls als nicht-GT-belegbar
  → Opus-Niedrigscores für Tool-Use sind strukturelles Messartefakt, kein Qualitätsmangel


/var/folders/z8/rlz8vq9j2cs6wj25qj_g6y700000gn/T/ipykernel_93790/2530672452.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7 · Methodische Einschränkungen

Die folgenden Punkte sind bei der Interpretation der Ergebnisse zu berücksichtigen:

| # | Einschränkung | Konsequenz |
|---|---|---|
| 1 | **Ceiling-Effekt**: Judge vergibt fast ausschließlich 4–5/5 | Clarity und Completeness diskriminieren nicht zwischen Pipelines; Radar-Diagramm suggeriert mehr Differenzierung als vorhanden |
| 2 | **Self-Preference-Bias**: Judge-Modell = Generation-Modell (`claude-sonnet-4-6`) | Mögliche Bevorzugung des eigenen Stils; literaturbekannter Bias (Self-Preference) nicht ausschließbar |
| 3 | **n = 10 pro Pipeline** (5 Instanzen × 2 Modelle) | Geringe statistische Power; Mittelwertunterschiede zwischen Pipelines nicht inferenzstatistisch absicherbar |
| 4 | **Kein Repeated Sampling** | LLM-Variabilität nicht gemessen; Consistency-Metrik fehlt trotz ursprünglicher Planung |
| 5 | **Convenience-Sampling**: Instanzen [42, 100, 250, 500, 1337] | Nicht systematisch nach cnt-Quantilen gewählt; repräsentiert niedrige bis hohe Nachfrage, aber kein stratifiziertes Design |
| 6 | **Zufälliger Train/Test-Split** bei Zeitreihendaten | Mögliche temporale Leakage; R² > 0,95 teilweise dadurch erklärbar; für XAI-Demonstration vertretbar, aber Generalisierungsaussagen sind eingeschränkt |
| 7 | **Selection-Bias Ichmoukhamedov-Metriken** (→ Notebook 08) | RA/SA/VA berechnen sich nur über Features, die das LLM selbst erwähnt hat; unerwähnte Top-K-Features werden nicht bestraft → metrisches Ergebnis überschätzt echte Faithfulness |
| 8 | **Judge-Prompt ohne Kalibrierungs-Rubrik** | Keine Beispiele für Scores 1–3 im Prompt; führt zu mildem Urteil und Ceiling-Effekt (→ verbesserter Prompt als zukünftige Arbeit) |

**Fazit für die Belegarbeit:**  
Die Ergebnisse sind als explorative Demonstration zu verstehen. Statistisch robuste Aussagen über Pipeline-Unterschiede erfordern größere Stichproben (n ≥ 30 pro Pipeline), einen unabhängigen Judge und Repeated Sampling.


In [17]:
# Ceiling-Effekt quantifizieren: Wie viele Scores sind = 5?
if len(judge_df) > 0:
    total_valid = judge_df[['faithfulness', 'clarity', 'completeness']].count().sum()
    total_5     = (judge_df[['faithfulness', 'clarity', 'completeness']] == 5).sum().sum()
    total_4     = (judge_df[['faithfulness', 'clarity', 'completeness']] == 4).sum().sum()
    print(f"Ceiling-Effekt: {total_5}/{total_valid} Scores = 5 ({100*total_5/total_valid:.0f}%)")
    print(f"               {total_4}/{total_valid} Scores = 4 ({100*total_4/total_valid:.0f}%)")
    print(f"               {total_valid - total_4 - total_5} Scores ≤ 3")
    print()
    print("Std-Abweichungen der Judge-Scores pro Pipeline:")
    display(
        judge_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']]
        .std().round(3).rename(columns=lambda c: c + '_std')
    )


Ceiling-Effekt: 114/159 Scores = 5 (72%)
               42/159 Scores = 4 (26%)
               3 Scores ≤ 3

Std-Abweichungen der Judge-Scores pro Pipeline:


,faithfulness_std,clarity_std,completeness_std
pipeline_label,,,
JSON→Text,0.605,0.410,0.224
Tool-Use,0.500,0.512,0.000
Vision,0.664,0.437,0.332
